# Exercise A: Discover the Asta Tools
1. Set ASTA_API_KEY to the class key.
2. Write a Python script that sends an HTTP POST to the Asta MCP endpoint with a tools/list JSON-RPC message.
3. Parse the response and print each tool's name, a one-line description, and its required parameters with types.
4. In a comment at the top of your file, answer: Which tool would you use to find all papers about "transformer attention mechanisms"? Which would you use to find who else published in the same area as a specific author?

In [2]:
import requests, os, json

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/event-stream",
    "x-api-key": os.environ["ASTA_API_KEY"]
}
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}
resp = requests.post(
    "https://asta-tools.allen.ai/mcp/v1",
    headers=headers,
    json=payload
)

# SSE response: extract the line starting with "data:"
data_line = next(l[6:] for l in resp.text.splitlines() if l.startswith("data:"))
tools = json.loads(data_line)["result"]["tools"]

for tool in tools:
    name = tool["name"]
    description = tool.get("description", "").strip().split("\n")[0]
    schema = tool.get("inputSchema", {})
    props = schema.get("properties", {})
    required = set(schema.get("required", []))

    required_params = [f"{p} ({props[p].get('type', 'any')})" for p in props if p in required]
    optional_params = [f"{p} ({props[p].get('type', 'any')})" for p in props if p not in required]

    print(f"Tool: {name}")
    print(f"  Description: {description}")
    print(f"  Required: {', '.join(required_params) if required_params else '(none)'}")
    if optional_params:
        print(f"  Optional: {', '.join(optional_params)}")
    print()


Tool: get_paper
  Description: Get details about a paper by its id.
  Required: paper_id (string)
  Optional: fields (string)

Tool: get_paper_batch
  Description: Get details about a list of papers by their ids.
  Required: ids (array)
  Optional: fields (string)

Tool: get_citations
  Description: Get details about the papers that cite this paper (i.e. papers in whose bibliography this paper appears)
  Required: paper_id (string)
  Optional: fields (string), limit (integer), publication_date_range (string)

Tool: search_authors_by_name
  Description: Search for authors by name.
  Required: name (string)
  Optional: fields (string), limit (integer)

Tool: get_author_papers
  Description: Get papers written by this author.
  Required: author_id (string)
  Optional: paper_fields (string), limit (integer), publication_date_range (string)

Tool: search_papers_by_relevance
  Description: Search for papers by keyword relevance.
  Required: keyword (string)
  Optional: fields (string), limit

Which tool would you use to find all papers about "transformer attention mechanisms"?

search_papers_by_relevance to search for papers by keyword relevance

Which would you use to find who else published in the same area as a specific author?

get_author_papers would get papers written by a specific author, and then those papers could be fed into get_citations about papers that cite those papers -- since we'd assume that papers that cite each other are published in the same area / around a similar topic. 

# Exercise B: Direct Asta Tool Calls -- Three Focused Drills

## Drill 1 -- search_papers: Find recent LLM agent papers
Write a function that calls search_papers with the query "large language model agents", requests the fields title,abstract,year,authors, and prints the top 5 results as a numbered list with title and year.

In [3]:
def search_papers(query, fields="title,abstract,year,authors", limit=5):
    url = "https://asta-tools.allen.ai/mcp/v1"
    payload = {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": {
            "name": "search_papers_by_relevance",
            "arguments": {
                "keyword": query,
                "fields": fields,
                "limit": limit
            }
        }
    }
    resp = requests.post(url, headers=headers, json=payload)
    data_line = next(l[6:] for l in resp.text.splitlines() if l.startswith("data:"))
    content_blocks = json.loads(data_line)["result"]["content"]

    for i, block in enumerate(content_blocks, 1):
        paper = json.loads(block["text"])
        title = paper.get("title", "N/A")
        year = paper.get("year", "N/A")
        print(f"{i}. {title} ({year})")

search_papers("large language model agents")


1. InjecAgent: Benchmarking Indirect Prompt Injections in Tool-Integrated Large Language Model Agents (2024)
2. Memory-R1: Enhancing Large Language Model Agents to Manage and Utilize Memories via Reinforcement Learning (2025)
3. A Survey of Large Language Model Agents for Question Answering (2025)
4. Large language model agents can use tools to perform clinical calculations (2025)
5. Emergence of human-like polarization among large language model agents (2025)


## Drill 2 -- get_citations: Trace impact of a landamrk paper
Call get_citations for the original BERT paper (ARXIV:1810.04805), filtered to papers from 2023 onward. Print the count of results and the titles of the first 5 citing papers.

In [4]:
# "arguments": {
#     "paper_id": "ARXIV:1810.04805",
#     "fields": "title,year,authors",
#     "limit": 10,
#     "publication_date_range": "2023-01-01:"
# }

In [5]:
url = "https://asta-tools.allen.ai/mcp/v1"
payload = {
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {
        "name": "get_citations",
        "arguments": {
            "paper_id": "ARXIV:2210.03629",
            "fields": "title,year,authors"
        }
    }
}
resp = requests.post(url, headers=headers, json=payload)
data_line = next(l[6:] for l in resp.text.splitlines() if l.startswith("data:"))
content_blocks = json.loads(data_line)["result"]["content"]

papers = [json.loads(block["text"])["citingPaper"] for block in content_blocks]
papers.sort(key=lambda p: p.get("year") or 0)

print(f"Count: {len(papers)}\n")
for i, paper in enumerate(papers, 1):
    print(f"{i}. {paper.get('title', 'N/A')} ({paper.get('year', 'N/A')})")


Count: 100

1. Enhancing large language models for knowledge graph question answering via multi-granularity knowledge injection and structured reasoning path-augmented prompting (2026)
2. Neuro-symbolic agentic AI: Architectures, integration patterns, applications, open challenges and future research directions (2026)
3. ChatPRE: Knowledge-aware protocol analysis with LLMs for intelligent segmentation (2026)
4. INKER: Adaptive dynamic retrieval augmented generation with internal-external knowledge integration (2026)
5. Vision-Language Model-Driven Human-Vehicle Interaction for Autonomous Driving: Status, Challenge, and Innovation (2026)
6. Why Agents Compromise Safety Under Pressure (2026)
7. Universe Routing: Why Self-Evolving Agents Need Epistemic Control (2026)
8. Financial Transaction Retrieval and Contextual Evidence for Knowledge-Grounded Reasoning (2026)
9. Confusion-Aware In-Context-Learning for Vision-Language Models in Robotic Manipulation (2026)
10. QiboAgent: a practitioner

## Drill 3 -- get_references: Understand a paper's intellectual foundation
Call get_references for the ReAct paper (ARXIV:2210.03629) and print the titles and years of its references, sorted by year ascending. This gives you a snapshot of the intellectual lineage of the paper.

In [6]:
url = "https://asta-tools.allen.ai/mcp/v1"
payload = {
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "name": "get_paper",
        "arguments": {
            "paper_id": "ARXIV:2210.03629",
            "fields": "references.title,references.year"
        }
    }
}
resp = requests.post(url, headers=headers, json=payload)
data_line = next(l[6:] for l in resp.text.splitlines() if l.startswith("data:"))
content_blocks = json.loads(data_line)["result"]["content"]
paper = json.loads(content_blocks[0]["text"])
references = paper.get("references", [])

references_sorted = sorted(references, key=lambda r: r.get("year") or 0)

print(f"Count: {len(references_sorted)}\n")
for i, ref in enumerate(references_sorted, 1):
    print(f"{i}. {ref.get('title', 'N/A')} ({ref.get('year', 'N/A')})")

Count: 63

1. Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture? (None)
2. put knife 1 in/on countertop 1 Nothing (None)
3. need to search Adam Clayton Powell and The Saimaa Gesture, and find which documentary is about Finnish rock groups (None)
4. Action 2: Search[Linda Hart (None)
5. Observation 1 Could not find [Beautiful] (None)
6. Result 1 (None)
7. Milhouse Mussolini Van Houten (None)
8. Thought 2: I need to search Frigg instead (None)
9. Musician and satirist Allie (None)
10. I need to search Nicholas Ray and Elia Kazan, find their professions, then find the profession they have in common (None)
11. Question Which magazine was started first Arthurâs Magazine or First for Women? Thought 1 I need to search Arthurâs Magazine and First for Women, and find which was started first (None)
12. Observation 2 (Result 1 / 1) Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous (None)
13. Vanderbilt University was f

In [42]:
url = "https://asta-tools.allen.ai/mcp/v1"
payload = {
    "jsonrpc": "2.0",
    "id": 4,
    "method": "tools/call",
    "params": {
        "name": "get_paper",
        "arguments": {
            "paper_id": "ARXIV:2210.03629",
            "fields": "references"
        }
    }
}
resp = requests.post(url, headers=headers, json=payload)
data_line = next(l[6:] for l in resp.text.splitlines() if l.startswith("data:"))
content_blocks = json.loads(data_line)["result"]["content"]
paper = json.loads(content_blocks[0]["text"])
references = paper.get("references", [])
print(references[0])

{'paperId': '74eae12620bd1c1393e268bddcb6f129a5025166', 'title': 'Improving alignment of dialogue agents via targeted human judgements'}


# Exercise C: Asta-Powered Research Chatbot

